
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.6_flash_attention_algorithm/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.6_flash_attention_algorithm/lab.ipynb)

# Lab 3.6: Flash Attention Algorithm — Tiled Attention with Online Softmax

Module 3.5 showed standard attention uses O(N²) memory. Here we fix it.


In [ ]:
# Cell 1: Setup — verify GPU available
import torch
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
assert device == "cuda", "This lab requires a GPU for memory measurement." 


In [ ]:
# Cell 2: Standard attention — materializes full N×N matrix (O(N²) memory)
def standard_attention(Q, K, V):
    """Textbook attention: compute full score matrix, softmax, multiply V."""
    d_k = Q.shape[-1]
    # S is [B, H, N, N] — this is the O(N²) memory bottleneck
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V)


In [ ]:
# Cell 3: Tiled attention with online softmax — O(N) memory
# Key insight: we never materialize the full N×N matrix.
# Instead, process K,V in tiles, maintaining running max and sum for numerically stable softmax.

def tiled_attention(Q, K, V, block_size=64):
    """Flash-attention-style tiled computation. Memory is O(N) not O(N²)."""
    B, H, N, d = Q.shape
    # Output accumulator and softmax normalization stats
    O = torch.zeros_like(Q)          # [B, H, N, d]
    l = torch.zeros(B, H, N, 1, device=Q.device)  # running sum of exp
    m = torch.full((B, H, N, 1), float('-inf'), device=Q.device)  # running max

    # Tile over key/value sequence dimension
    for j_start in range(0, N, block_size):
        j_end = min(j_start + block_size, N)
        # Load one tile of K, V — size [B, H, block, d]
        Kj = K[:, :, j_start:j_end, :]
        Vj = V[:, :, j_start:j_end, :]

        # Compute partial scores for this tile: [B, H, N, block]
        s = torch.matmul(Q, Kj.transpose(-2, -1)) / math.sqrt(d)

        # Online softmax update (numerically stable)
        m_new = torch.maximum(m, s.max(dim=-1, keepdim=True).values)
        # Rescale old accumulator to new max
        exp_old = torch.exp(m - m_new)
        # Compute exp of current tile scores under new max
        exp_s = torch.exp(s - m_new)

        # Update running sum and output
        l = l * exp_old + exp_s.sum(dim=-1, keepdim=True)
        O = O * exp_old + torch.matmul(exp_s, Vj)
        m = m_new

    # Final normalization
    return O / l


In [ ]:
# Cell 4: Correctness verification — compare against PyTorch SDPA
torch.manual_seed(42)
B, H, N, d = 2, 8, 512, 64  # batch, heads, seq_len, head_dim
Q = torch.randn(B, H, N, d, device=device)
K = torch.randn(B, H, N, d, device=device)
V = torch.randn(B, H, N, d, device=device)

# Reference: PyTorch's scaled_dot_product_attention
ref = F.scaled_dot_product_attention(Q, K, V)
# Our tiled implementation
out = tiled_attention(Q, K, V, block_size=64)

max_diff = (ref - out).abs().max().item()
print(f"Max absolute difference vs torch SDPA: {max_diff:.2e}")
assert max_diff < 1e-5, f"FAILED: diff {max_diff} exceeds 1e-5"
print("✓ Correctness verified (max diff < 1e-5)")


In [ ]:
# Cell 5: Also verify against our standard_attention implementation
ref_std = standard_attention(Q, K, V)
max_diff_std = (ref_std - out).abs().max().item()
print(f"Max diff vs standard_attention: {max_diff_std:.2e}")
assert max_diff_std < 1e-5
print("✓ Both implementations agree")


In [ ]:
# Cell 6: Memory measurement — standard vs tiled
# We measure peak GPU memory allocated during forward pass.

def measure_peak_memory(fn, Q, K, V):
    """Run fn and return peak memory allocated in MB."""
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    baseline = torch.cuda.memory_allocated()
    _ = fn(Q, K, V)
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    return (peak - baseline) / (1024**2)  # MB

# Test across increasing sequence lengths
seq_lengths = [256, 512, 1024, 2048, 4096]
mem_standard = []
mem_tiled = []

for N in seq_lengths:
    Q = torch.randn(1, 8, N, 64, device=device)
    K = torch.randn(1, 8, N, 64, device=device)
    V = torch.randn(1, 8, N, 64, device=device)

    mem_std = measure_peak_memory(standard_attention, Q, K, V)
    mem_standard.append(mem_std)

    mem_til = measure_peak_memory(lambda q,k,v: tiled_attention(q,k,v, block_size=64), Q, K, V)
    mem_tiled.append(mem_til)

    print(f"N={N:5d} | Standard: {mem_std:8.1f} MB | Tiled: {mem_til:8.1f} MB | Ratio: {mem_std/max(mem_til,0.01):.1f}x")

# Clean up
del Q, K, V
torch.cuda.empty_cache()


In [ ]:
# Cell 7: Plot memory scaling — O(N²) vs O(N)
fig_6, ax_6 = plt.subplots(1, 1, figsize=(8, 5))

ax_6.plot(seq_lengths, mem_standard, 'o-', color='#e74c3c', linewidth=2, label='Standard (O(N²) memory)')
ax_6.plot(seq_lengths, mem_tiled, 's-', color='#2ecc71', linewidth=2, label='Tiled (O(N) memory)')

ax_6.set_xlabel('Sequence Length (N)', fontsize=12)
ax_6.set_ylabel('Peak Memory (MB)', fontsize=12)
ax_6.set_title('Attention Memory Scaling: Standard vs Tiled', fontsize=14)
ax_6.legend(fontsize=11)
ax_6.grid(True, alpha=0.3)
ax_6.set_xticks(seq_lengths)

plt.tight_layout()
plt.savefig('memory_scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: memory_scaling.png")


In [ ]:
# Cell 8: Verify quadratic vs linear scaling numerically
# Fit power law: memory ∝ N^α. Standard should give α≈2, tiled α≈1.
import numpy as np

log_n = np.log(seq_lengths)
log_std = np.log(mem_standard)
log_til = np.log(mem_tiled)

# Linear regression in log-log space gives the exponent
alpha_std = np.polyfit(log_n, log_std, 1)[0]
alpha_til = np.polyfit(log_n, log_til, 1)[0]

print(f"Standard attention memory exponent: {alpha_std:.2f} (expected ≈2.0)")
print(f"Tiled attention memory exponent:    {alpha_til:.2f} (expected ≈1.0)")
print(f"\nConclusion: Tiled attention reduces memory complexity from O(N^{alpha_std:.1f}) to O(N^{alpha_til:.1f})")
